In [3]:
import pandas as pd
import tkinter as tk
from tkinter import scrolledtext, messagebox
import os

# ================= CONFIG =================
INPUT_PATH = r"C:\Users\Admin\OneDrive\Documents\Câu hỏi.xlsx"
OUTPUT_PATH = "review_checked.xlsx"

COL_CHATGPT = "Nhận xét ChatGPT"
COL_AILUAT = "Nhận xét AILuat Chatbot"
COL_CHECKED = "Checked"

# IRAC Score columns with weights
IRAC_COMPONENTS = [
    ('Issue', 0.10),
    ('Rule', 0.35),
    ('Application', 0.40),
    ('Conclusion', 0.10),
    ('Clarity', 0.05)
]

COL_CGPT_IRAC = {comp[0]: f'ChatGPT_{comp[0]}' for comp in IRAC_COMPONENTS}
COL_CGPT_IRAC['Total'] = 'ChatGPT_Total_Score'

COL_AILUAT_IRAC = {comp[0]: f'AILuat_{comp[0]}' for comp in IRAC_COMPONENTS}
COL_AILUAT_IRAC['Total'] = 'AILuat_Total_Score'

BTN_DEFAULT = "#f0f0f0"
BTN_OK = "#6CCF8E"
BTN_NOT_OK = "#F28B82"

# ================= LOAD DATA =================
if os.path.exists(OUTPUT_PATH):
    df = pd.read_excel(OUTPUT_PATH)
else:
    df = pd.read_excel(INPUT_PATH)

for col in [COL_CHATGPT, COL_AILUAT, COL_CHECKED]:
    if col not in df.columns:
        df[col] = ""

# Add IRAC columns if they don't exist
for cols in [COL_CGPT_IRAC, COL_AILUAT_IRAC]:
    for col in cols.values():
        if col not in df.columns:
            df[col] = ""

current_index = 0

# ================= HELPERS =================
def save():
    df.to_excel(OUTPUT_PATH, index=False)

def calculate_total_score(scores_dict):
    """Calculate weighted total score"""
    try:
        total = 0
        for i, (comp, weight) in enumerate(IRAC_COMPONENTS):
            score = scores_dict.get(comp, "")
            if score and str(score).strip():
                total += float(score) * weight
        return round(total, 2)
    except (ValueError, TypeError):
        return ""

def has_status(value):
    if not isinstance(value, str):
        return False
    return value.startswith("OK") or value.startswith("NOT OK")

def parse_value(value):
    if not isinstance(value, str):
        return "", ""

    value = value.strip()

    if value.startswith("OK"):
        parts = value.split("-", 1)
        return "OK", parts[1].strip() if len(parts) > 1 else ""

    if value.startswith("NOT OK"):
        parts = value.split("-", 1)
        return "NOT OK", parts[1].strip() if len(parts) > 1 else ""

    return "", ""

def build_value(status, comment):
    comment = comment.strip()
    return f"{status} - {comment}" if comment else status

def update_checked(idx):
    if has_status(df.at[idx, COL_CHATGPT]) and has_status(df.at[idx, COL_AILUAT]):
        df.at[idx, COL_CHECKED] = "DONE"
        btn_next.config(state=tk.NORMAL)
    else:
        df.at[idx, COL_CHECKED] = ""
        btn_next.config(state=tk.DISABLED)

def reset_buttons():
    for b in [btn_cgpt_ok, btn_cgpt_not, btn_ailuat_ok, btn_ailuat_not]:
        b.config(bg=BTN_DEFAULT)

def update_buttons(idx):
    reset_buttons()

    cgpt_status, _ = parse_value(df.at[idx, COL_CHATGPT])
    ailuat_status, _ = parse_value(df.at[idx, COL_AILUAT])

    if cgpt_status == "OK":
        btn_cgpt_ok.config(bg=BTN_OK)
    elif cgpt_status == "NOT OK":
        btn_cgpt_not.config(bg=BTN_NOT_OK)

    if ailuat_status == "OK":
        btn_ailuat_ok.config(bg=BTN_OK)
    elif ailuat_status == "NOT OK":
        btn_ailuat_not.config(bg=BTN_NOT_OK)

    update_checked(idx)

def fill_text(widget, value):
    widget.delete("1.0", tk.END)
    widget.insert(tk.END, "" if pd.isna(value) else str(value))

def fill_entry(widget, value):
    widget.delete(0, tk.END)
    widget.insert(0, "" if pd.isna(value) else str(value))

# ================= LOAD ROW =================
def load_row(idx):
    if idx >= len(df):
        save()
        messagebox.showinfo("Done", "All rows reviewed!")
        root.destroy()
        return

    row = df.loc[idx]

    fill_text(txt_question, row["Câu hỏi"])
    fill_text(txt_chatgpt, row["ChatGPT"])
    fill_text(txt_chatgpt_ref, row["legal_references (từ chat GPT)"])
    fill_text(txt_ailuat, row["AILuat ChatBot"])
    fill_text(txt_ailuat_ref, row["legal_references (từ AILuat Chatbot)"])

    cgpt_status, cgpt_cmt = parse_value(row[COL_CHATGPT])
    ailuat_status, ailuat_cmt = parse_value(row[COL_AILUAT])

    fill_text(entry_cgpt_comment, cgpt_cmt)
    fill_text(entry_ailuat_comment, ailuat_cmt)

    # Load IRAC scores
    for comp, _ in IRAC_COMPONENTS:
        fill_entry(irac_cgpt_entries[comp], row[COL_CGPT_IRAC[comp]])
        fill_entry(irac_ailuat_entries[comp], row[COL_AILUAT_IRAC[comp]])

    # Update total scores
    update_total_scores()

    lbl_index.config(text=f"Row {idx + 1}/{len(df)}")
    update_buttons(idx)

# ================= ACTIONS =================
def mark_chatgpt(status):
    _, comment = parse_value(df.at[current_index, COL_CHATGPT])
    df.at[current_index, COL_CHATGPT] = build_value(status, comment)
    save()
    update_buttons(current_index)

def mark_ailuat(status):
    _, comment = parse_value(df.at[current_index, COL_AILUAT])
    df.at[current_index, COL_AILUAT] = build_value(status, comment)
    save()
    update_buttons(current_index)

def on_comment_change(event=None):
    cgpt_status, _ = parse_value(df.at[current_index, COL_CHATGPT])
    ailuat_status, _ = parse_value(df.at[current_index, COL_AILUAT])

    if cgpt_status:
        df.at[current_index, COL_CHATGPT] = build_value(
            cgpt_status, entry_cgpt_comment.get("1.0", tk.END)
        )

    if ailuat_status:
        df.at[current_index, COL_AILUAT] = build_value(
            ailuat_status, entry_ailuat_comment.get("1.0", tk.END)
        )

    save()
    update_checked(current_index)

def update_total_scores():
    # Calculate ChatGPT total
    cgpt_scores = {comp: irac_cgpt_entries[comp].get() for comp, _ in IRAC_COMPONENTS}
    cgpt_total = calculate_total_score(cgpt_scores)
    if cgpt_total != '':
        lbl_cgpt_total.config(text=f"Final Score: {cgpt_total}/10", fg="blue", font=("Arial", 11, "bold"))
    else:
        lbl_cgpt_total.config(text="Final Score: -/10", fg="blue", font=("Arial", 11, "bold"))

    # Calculate AILuat total
    ailuat_scores = {comp: irac_ailuat_entries[comp].get() for comp, _ in IRAC_COMPONENTS}
    ailuat_total = calculate_total_score(ailuat_scores)
    if ailuat_total != '':
        lbl_ailuat_total.config(text=f"Final Score: {ailuat_total}/10", fg="green", font=("Arial", 11, "bold"))
    else:
        lbl_ailuat_total.config(text="Final Score: -/10", fg="green", font=("Arial", 11, "bold"))

def on_irac_change(event=None):
    # Save ChatGPT IRAC scores
    cgpt_scores = {}
    for comp, _ in IRAC_COMPONENTS:
        value = irac_cgpt_entries[comp].get()
        df.at[current_index, COL_CGPT_IRAC[comp]] = value
        cgpt_scores[comp] = value

    # Calculate and save ChatGPT total
    cgpt_total = calculate_total_score(cgpt_scores)
    df.at[current_index, COL_CGPT_IRAC['Total']] = cgpt_total

    # Save AILuat IRAC scores
    ailuat_scores = {}
    for comp, _ in IRAC_COMPONENTS:
        value = irac_ailuat_entries[comp].get()
        df.at[current_index, COL_AILUAT_IRAC[comp]] = value
        ailuat_scores[comp] = value

    # Calculate and save AILuat total
    ailuat_total = calculate_total_score(ailuat_scores)
    df.at[current_index, COL_AILUAT_IRAC['Total']] = ailuat_total

    # Update display
    update_total_scores()
    save()

def next_row():
    global current_index
    if btn_next["state"] == tk.NORMAL:
        current_index += 1
        load_row(current_index)

def prev_row():
    global current_index
    if current_index > 0:
        current_index -= 1
        load_row(current_index)

def on_close():
    on_comment_change()
    on_irac_change()
    save()
    root.destroy()

# ================= GUI =================
root = tk.Tk()
root.title("Legal QA Reviewer with IRAC Scoring")
root.geometry("1200x800")
root.protocol("WM_DELETE_WINDOW", on_close)

# Create main canvas and scrollbar
main_canvas = tk.Canvas(root)
scrollbar = tk.Scrollbar(root, orient="vertical", command=main_canvas.yview)
scrollable_frame = tk.Frame(main_canvas)

scrollable_frame.bind(
    "<Configure>",
    lambda e: main_canvas.configure(scrollregion=main_canvas.bbox("all"))
)

canvas_window = main_canvas.create_window((0, 0), window=scrollable_frame, anchor="nw")

# Make the frame fill the canvas width
def configure_canvas_width(event):
    canvas_width = event.width
    main_canvas.itemconfig(canvas_window, width=canvas_width)

main_canvas.bind("<Configure>", configure_canvas_width)
main_canvas.configure(yscrollcommand=scrollbar.set)

# Pack scrollbar and canvas
scrollbar.pack(side="right", fill="y")
main_canvas.pack(side="left", fill="both", expand=True)

# Bind mouse wheel to scroll
def _on_mousewheel(event):
    main_canvas.yview_scroll(int(-1*(event.delta/120)), "units")

main_canvas.bind_all("<MouseWheel>", _on_mousewheel)

lbl_index = tk.Label(scrollable_frame, font=("Arial", 12, "bold"))
lbl_index.pack(pady=5)

tk.Label(scrollable_frame, text="Câu hỏi", font=("Arial", 10, "bold")).pack(anchor="w")
txt_question = scrolledtext.ScrolledText(scrollable_frame, height=4)
txt_question.pack(fill=tk.X, padx=5)

# ================= ChatGPT Section =================
tk.Label(scrollable_frame, text="ChatGPT Answer", font=("Arial", 10, "bold")).pack(anchor="w")
txt_chatgpt = scrolledtext.ScrolledText(scrollable_frame, height=6)
txt_chatgpt.pack(fill=tk.X, padx=5)

tk.Label(scrollable_frame, text="ChatGPT Legal References", fg="blue").pack(anchor="w")
txt_chatgpt_ref = scrolledtext.ScrolledText(scrollable_frame, height=3)
txt_chatgpt_ref.pack(fill=tk.X, padx=5)

frame_cgpt = tk.Frame(scrollable_frame)
frame_cgpt.pack(pady=5)
btn_cgpt_ok = tk.Button(frame_cgpt, text="✅ ChatGPT OK", width=20,
                        command=lambda: mark_chatgpt("OK"))
btn_cgpt_ok.pack(side=tk.LEFT, padx=5)
btn_cgpt_not = tk.Button(frame_cgpt, text="❌ ChatGPT NOT OK", width=20,
                         command=lambda: mark_chatgpt("NOT OK"))
btn_cgpt_not.pack(side=tk.LEFT, padx=5)

tk.Label(scrollable_frame, text="Nhận xét ChatGPT").pack(anchor="w")
entry_cgpt_comment = scrolledtext.ScrolledText(scrollable_frame, height=2)
entry_cgpt_comment.pack(fill=tk.X, padx=5)
entry_cgpt_comment.bind("<KeyRelease>", on_comment_change)

# ChatGPT IRAC Scoring
frame_cgpt_irac_header = tk.Frame(scrollable_frame)
frame_cgpt_irac_header.pack(fill=tk.X, padx=5, pady=(5,0))
tk.Label(frame_cgpt_irac_header, text="ChatGPT IRAC Score (0-10)",
         font=("Arial", 9, "bold"), fg="blue").pack(side=tk.LEFT)
lbl_cgpt_total = tk.Label(frame_cgpt_irac_header, text="Final Score: -/10",
                          font=("Arial", 11, "bold"), fg="blue")
lbl_cgpt_total.pack(side=tk.RIGHT, padx=10)

frame_cgpt_irac = tk.Frame(scrollable_frame)
frame_cgpt_irac.pack(fill=tk.X, padx=5, pady=5)

irac_cgpt_entries = {}
for i, (comp, weight) in enumerate(IRAC_COMPONENTS):
    row = i // 3
    col = (i % 3) * 3

    label_text = f"{comp} ({int(weight*100)}%):"
    tk.Label(frame_cgpt_irac, text=label_text, width=18, anchor='e').grid(
        row=row, column=col, padx=(0,5), pady=2)
    entry = tk.Entry(frame_cgpt_irac, width=8)
    entry.grid(row=row, column=col+1, padx=(0,15), pady=2)
    entry.bind("<KeyRelease>", on_irac_change)
    irac_cgpt_entries[comp] = entry

# ================= AILuat Section =================
tk.Label(scrollable_frame, text="AILuat ChatBot Answer", font=("Arial", 10, "bold")).pack(anchor="w", pady=(10,0))
txt_ailuat = scrolledtext.ScrolledText(scrollable_frame, height=6)
txt_ailuat.pack(fill=tk.X, padx=5)

tk.Label(scrollable_frame, text="AILuat Legal References", fg="green").pack(anchor="w")
txt_ailuat_ref = scrolledtext.ScrolledText(scrollable_frame, height=3)
txt_ailuat_ref.pack(fill=tk.X, padx=5)

frame_ailuat = tk.Frame(scrollable_frame)
frame_ailuat.pack(pady=5)
btn_ailuat_ok = tk.Button(frame_ailuat, text="✅ AILuat OK", width=20,
                          command=lambda: mark_ailuat("OK"))
btn_ailuat_ok.pack(side=tk.LEFT, padx=5)
btn_ailuat_not = tk.Button(frame_ailuat, text="❌ AILuat NOT OK", width=20,
                           command=lambda: mark_ailuat("NOT OK"))
btn_ailuat_not.pack(side=tk.LEFT, padx=5)

tk.Label(scrollable_frame, text="Nhận xét AILuat Chatbot").pack(anchor="w")
entry_ailuat_comment = scrolledtext.ScrolledText(scrollable_frame, height=2)
entry_ailuat_comment.pack(fill=tk.X, padx=5)
entry_ailuat_comment.bind("<KeyRelease>", on_comment_change)

# AILuat IRAC Scoring
frame_ailuat_irac_header = tk.Frame(scrollable_frame)
frame_ailuat_irac_header.pack(fill=tk.X, padx=5, pady=(5,0))
tk.Label(frame_ailuat_irac_header, text="AILuat IRAC Score (0-10)",
         font=("Arial", 9, "bold"), fg="green").pack(side=tk.LEFT)
lbl_ailuat_total = tk.Label(frame_ailuat_irac_header, text="Final Score: -/10",
                            font=("Arial", 11, "bold"), fg="green")
lbl_ailuat_total.pack(side=tk.RIGHT, padx=10)

frame_ailuat_irac = tk.Frame(scrollable_frame)
frame_ailuat_irac.pack(fill=tk.X, padx=5, pady=5)

irac_ailuat_entries = {}
for i, (comp, weight) in enumerate(IRAC_COMPONENTS):
    row = i // 3
    col = (i % 3) * 3

    label_text = f"{comp} ({int(weight*100)}%):"
    tk.Label(frame_ailuat_irac, text=label_text, width=18, anchor='e').grid(
        row=row, column=col, padx=(0,5), pady=2)
    entry = tk.Entry(frame_ailuat_irac, width=8)
    entry.grid(row=row, column=col+1, padx=(0,15), pady=2)
    entry.bind("<KeyRelease>", on_irac_change)
    irac_ailuat_entries[comp] = entry

# ================= Navigation =================
nav = tk.Frame(scrollable_frame)
nav.pack(pady=10)
tk.Button(nav, text="⬅ Previous", width=18, command=prev_row).pack(side=tk.LEFT, padx=10)
btn_next = tk.Button(nav, text="➡ Next", width=18, command=next_row, state=tk.DISABLED)
btn_next.pack(side=tk.LEFT, padx=10)

load_row(current_index)
root.mainloop()

Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\tkinter\__init__.py", line 2068, in __call__
    return self.func(*args)
           ~~~~~~~~~^^^^^^^
  File "C:\Users\Admin\AppData\Local\Temp\ipykernel_3336\3252261770.py", line 250, in on_close
    on_comment_change()
    ~~~~~~~~~~~~~~~~~^^
  File "C:\Users\Admin\AppData\Local\Temp\ipykernel_3336\3252261770.py", line 190, in on_comment_change
    save()
    ~~~~^^
  File "C:\Users\Admin\AppData\Local\Temp\ipykernel_3336\3252261770.py", line 53, in save
    df.to_excel(OUTPUT_PATH, index=False)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\util\_decorators.py", line 333, in wrapper
    return func(*args, **kwargs)
  File "C:\Users\Admin\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\core\generic.py", line 2439, in to_excel
    formatter.write(
    ~~~~~~